# Module 13 Guided Lab: Classification Models

## Logistic Regression and Decision Trees

**BAN 6003: Data Management and Analytics Integration**

In Module 12, we used linear regression to predict a continuous outcome. This week, we move to classification.

The main idea is:

> Classification models predict categories or classes.

For this lab, we will use `nycflights13` to predict whether a flight arrives late.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Distinguish regression problems from classification problems.
2. Create a binary classification target.
3. Fit a logistic regression model.
4. Fit a basic decision tree classifier.
5. Generate class predictions.
6. Review a confusion matrix.
7. Calculate accuracy, precision, and recall.
8. Compare model outputs and suitability.

This is an introductory classification lab. The goal is to learn the workflow, not to build the perfect model.

## Business Scenario

Imagine an airline operations team wants to flag flights that are likely to arrive late.

A simple classification question could be:

> Will this flight arrive at least 15 minutes late?

For this lab:

- **Target:** `arr_late_15`
- **Class 1:** flight arrived 15 or more minutes late
- **Class 0:** flight did not arrive 15 or more minutes late

We will use a small set of numeric features to keep the workflow clear.

## Important Timing Note

Some features in this teaching example may not be available before a real-time prediction.

For example:

- `dep_delay` is known after departure.
- `air_time` may only be fully known after the flight.

So this lab is mainly for learning the classification workflow. In a real project, you must check whether features are available at the time the prediction is made.

## 0. Setup

### Package setup note

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are running the repository on your **local computer** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.



In [ ]:
# Codespaces: nycflights13 and scikit-learn are installed from requirements.txt.
# Local only: if the imports below fail, uncomment and run the needed line(s):
# %pip install nycflights13
# %pip install -q scikit-learn

import nycflights13
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)


## 1. Load and Prepare the Data

We start with a flight-level table.

One row represents one flight.

In [ ]:
flights = nycflights13.flights.copy()

classification_data = flights[[
    "month",
    "dep_delay",
    "arr_delay",
    "distance",
    "air_time"
]].copy()

classification_data.head()

## 2. Create a Binary Target

The original `arr_delay` column is continuous because it measures minutes.

For classification, we create a binary target:

```python
arr_late_15 = 1 if arr_delay >= 15
arr_late_15 = 0 otherwise
```

This converts the problem from regression to classification.

In [ ]:
classification_data = classification_data.dropna().copy()

classification_data["arr_late_15"] = (classification_data["arr_delay"] >= 15).astype(int)

classification_data[["arr_delay", "arr_late_15"]].head(10)

In [ ]:
classification_data["arr_late_15"].value_counts()

In [ ]:
classification_data["arr_late_15"].value_counts(normalize=True)

### Class balance

Before fitting a classification model, check the class balance.

If one class is much more common than the other, accuracy can be misleading.

### Your Turn 1

Write one sentence explaining what class 1 and class 0 mean in this lab.

**Your answer:**  
Type your answer here.

## 3. Define Features and Target

We use:

- `X`: feature variables
- `y`: target variable

For this lab, the features are numeric to keep the model simple.

In [ ]:
features = ["dep_delay", "distance", "air_time", "month"]
target = "arr_late_15"

X = classification_data[features]
y = classification_data[target]

X.head()

In [ ]:
y.head()

## 4. Train/Test Split

As in linear regression, we split data into training and testing sets.

We use `stratify=y` to keep the class proportions similar in the training and testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

## 5. Logistic Regression

Logistic regression is used for classification, even though the name includes "regression."

For binary classification, it estimates the probability that an observation belongs to class 1.

Here, class 1 means the flight arrived at least 15 minutes late.

In [ ]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

print("Logistic regression model fitted.")

## 6. Logistic Regression Predictions

We can generate:

- predicted classes using `.predict()`
- predicted probabilities using `.predict_proba()`

The predicted class is usually based on a probability threshold of 0.5.

In [ ]:
log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:, 1]

log_results = pd.DataFrame({
    "actual": y_test,
    "predicted_class": log_pred,
    "predicted_probability_late": log_prob
})

log_results.head()

### Coefficients

A positive coefficient means the feature is associated with a higher predicted probability of class 1, holding other features constant.

Be cautious: this is not automatically causal.

In [ ]:
log_coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": log_model.coef_[0]
}).sort_values("coefficient", ascending=False)

log_coefficients

### Your Turn 2

Which feature has the largest positive coefficient in the logistic regression model?

Write one cautious interpretation.

**Your answer:**  
Type your answer here.

## 7. Confusion Matrix and Metrics

A confusion matrix compares actual classes with predicted classes.

For binary classification, it includes:

- True negatives
- False positives
- False negatives
- True positives

In [ ]:
cm_log = confusion_matrix(y_test, log_pred)
cm_log

In [ ]:
cm_log_df = pd.DataFrame(
    cm_log,
    index=["Actual 0: not late", "Actual 1: late"],
    columns=["Predicted 0: not late", "Predicted 1: late"]
)

cm_log_df

### Classification metrics

We will calculate:

- **Accuracy:** overall proportion correct
- **Precision:** among predicted late flights, how many were actually late?
- **Recall:** among actually late flights, how many did the model catch?

In [ ]:
log_accuracy = accuracy_score(y_test, log_pred)
log_precision = precision_score(y_test, log_pred)
log_recall = recall_score(y_test, log_pred)

pd.DataFrame({
    "model": ["Logistic Regression"],
    "accuracy": [log_accuracy],
    "precision": [log_precision],
    "recall": [log_recall]
})

In [ ]:
print(classification_report(y_test, log_pred))

### Your Turn 3

Write 2–3 sentences interpreting the logistic regression model.

Use plain language. Mention at least one metric.

**Your answer:**  
Type your answer here.

## 8. Decision Tree Classifier

A decision tree uses a sequence of rules to classify observations.

It is often more intuitive than logistic regression because the model can be visualized as a tree.

To keep the tree readable, we set `max_depth=3`.

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_model.fit(X_train, y_train)

print("Decision tree model fitted.")

In [ ]:
tree_pred = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_precision = precision_score(y_test, tree_pred)
tree_recall = recall_score(y_test, tree_pred)

pd.DataFrame({
    "model": ["Decision Tree"],
    "accuracy": [tree_accuracy],
    "precision": [tree_precision],
    "recall": [tree_recall]
})

## 9. Visualize the Decision Tree

The visualization shows the rules the tree learned.

Each split asks a question about one feature.

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(
    tree_model,
    feature_names=features,
    class_names=["Not late", "Late"],
    filled=False,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree Classifier")
plt.show()

### Your Turn 4

Look at the first split in the tree.

What feature does the tree use first? What does that suggest about the classification task?

**Your answer:**  
Type your answer here.

## 10. Compare Logistic Regression and Decision Tree

Now compare both models in one table.

In [ ]:
comparison = pd.DataFrame({
    "model": ["Logistic Regression", "Decision Tree"],
    "accuracy": [log_accuracy, tree_accuracy],
    "precision": [log_precision, tree_precision],
    "recall": [log_recall, tree_recall]
})

comparison

## 11. Model Suitability

Both models can be useful, but they have different strengths.

**Logistic regression:**

- simple and widely used
- gives coefficients
- works well for linear patterns
- less intuitive for some audiences

**Decision tree:**

- easy to explain as rules
- can capture non-linear patterns
- can overfit if the tree is too deep
- may be less stable than simpler models

In a business setting, model choice depends on performance, interpretability, and the decision context.

## 12. Final Practice: Build a Classification Model with a Different Target

Create a new binary target:

```python
dep_late_15 = 1 if dep_delay >= 15
dep_late_15 = 0 otherwise
```

Then fit either logistic regression or a decision tree to predict `dep_late_15`.

Use features that do not include `dep_delay`, because `dep_delay` defines the target.

In [ ]:
# Final Practice Step 1
# Create the dep_late_15 target.

In [ ]:
# Final Practice Step 2
# Choose features and target.

In [ ]:
# Final Practice Step 3
# Split the data.

In [ ]:
# Final Practice Step 4
# Fit a classification model.

In [ ]:
# Final Practice Step 5
# Generate predictions and calculate accuracy, precision, and recall.

### Final Reflection

Write 5–7 sentences answering:

1. How is classification different from regression?
2. What was the target variable in the main lab?
3. What did the confusion matrix show?
4. Which metric was easiest to understand?
5. Which model seemed more interpretable?
6. How might classification apply to your project?

**Your reflection:**  
Type your answer here.

## 13. Save Your Work

Before submitting:

1. Save this notebook.
2. Make sure all cells run.
3. Complete all Your Turn sections.
4. Complete the final practice and reflection.
5. Commit and push your work.

Suggested commands:

```bash
git add .
git commit -m "Complete Module 13 classification lab"
git push
```